# Bank Marketing — Model Comparison

## Objective

This notebook compares multiple classification models using
stratified cross-validation on the training set.

The held-out test set is not used for model selection or
hyperparameter tuning. It will be used only after the final
model configuration has been selected.

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv(
    "../data/raw/bank-full.csv",
    sep=";",
)

X = df.drop(columns=["y", "duration"]).copy()

y = df["y"].map({
    "no": 0,
    "yes": 1,
})

In [3]:
X["previously_contacted"] = (
    X["pdays"] != -1
).astype("int8")

X["pdays"] = X["pdays"].replace(-1, 0)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

In [5]:
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

binary_features = [
    "previously_contacted",
]

In [6]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ]
)

## Logistic Regression Comparison

In [8]:
models = {
    "logistic_standard": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "logistic_balanced": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    ),
}

In [9]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [10]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1_score": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

In [11]:
comparison_rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cross_validation,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise",
    )

    row = {
        "model": model_name,
    }

    for metric_name in scoring:
        metric_values = scores[f"test_{metric_name}"]

        row[f"{metric_name}_mean"] = metric_values.mean()
        row[f"{metric_name}_std"] = metric_values.std()

    comparison_rows.append(row)

In [12]:
logistic_comparison = (
    pd.DataFrame(comparison_rows)
    .set_index("model")
)

logistic_comparison.T

model,logistic_standard,logistic_balanced
accuracy_mean,0.892198,0.755087
accuracy_std,0.000609,0.005478
precision_mean,0.644981,0.266195
precision_std,0.012500,0.004007
recall_mean,0.175139,0.622077
recall_std,0.010938,0.009948
f1_score_mean,0.275222,0.372787
f1_score_std,0.012816,0.003332
roc_auc_mean,0.762780,0.763704
roc_auc_std,0.003060,0.003171


### Logistic Regression Cross-Validation Findings

- Standard Logistic Regression achieves a mean precision of 64.50% but identifies only 17.51% of the positive class.
- Balanced Logistic Regression increases mean recall to 62.21%, while mean precision decreases to 26.62%.
- The balanced model obtains a higher mean F1-score: 0.373 compared with 0.275.
- Mean accuracy decreases from 89.22% to 75.51% because the balanced model predicts more customers as potential subscribers.
- ROC-AUC and average precision remain nearly unchanged. This suggests that class weighting primarily changes the classification behavior at the default threshold rather than substantially improving probability ranking.
- Selecting between the two configurations requires considering the operational cost of missed subscribers and unnecessary customer contacts.

In [13]:
tree_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

tree_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

In [14]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            tree_categorical_transformer,
            categorical_features,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ]
)

## Tree-Based Model Comparison

In [15]:
tree_models = {
    "random_forest": Pipeline(
        steps=[
            (
                "preprocessor",
                tree_preprocessor,
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=200,
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=1,
                ),
            ),
        ]
    ),
    "hist_gradient_boosting": Pipeline(
        steps=[
            (
                "preprocessor",
                tree_preprocessor,
            ),
            (
                "classifier",
                HistGradientBoostingClassifier(
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    ),
}

In [16]:
tree_comparison_rows = []

for model_name, model in tree_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cross_validation,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise",
    )

    row = {
        "model": model_name,
    }

    for metric_name in scoring:
        metric_values = scores[f"test_{metric_name}"]

        row[f"{metric_name}_mean"] = metric_values.mean()
        row[f"{metric_name}_std"] = metric_values.std()

    tree_comparison_rows.append(row)

In [17]:
tree_comparison = (
    pd.DataFrame(tree_comparison_rows)
    .set_index("model")
)

model_comparison = pd.concat([
    logistic_comparison,
    tree_comparison,
])

model_comparison.T

model,logistic_standard,logistic_balanced,random_forest,hist_gradient_boosting
accuracy_mean,0.892198,0.755087,0.892723,0.825398
accuracy_std,0.000609,0.005478,0.001035,0.008151
precision_mean,0.644981,0.266195,0.621955,0.357762
precision_std,0.012500,0.004007,0.009567,0.013705
recall_mean,0.175139,0.622077,0.211297,0.615696
recall_std,0.010938,0.009948,0.010863,0.008134
f1_score_mean,0.275222,0.372787,0.315324,0.452328
f1_score_std,0.012816,0.003332,0.012915,0.009961
roc_auc_mean,0.762780,0.763704,0.778890,0.797301
roc_auc_std,0.003060,0.003171,0.004497,0.001565


## Model Comparison Findings

- Standard Logistic Regression produces relatively high precision but identifies only 17.51% of subscribers.
- Balanced Logistic Regression increases recall to 62.21%, but precision decreases to 26.62%.
- Random Forest improves probability ranking compared with both Logistic Regression configurations, reaching a mean ROC-AUC of 0.779 and a mean average precision of 0.417. However, its recall at the default threshold remains low at 21.13%.
- HistGradientBoosting obtains the best overall cross-validation results. It reaches a mean F1-score of 0.452, mean ROC-AUC of 0.797, and mean average precision of 0.443.
- HistGradientBoosting identifies 61.57% of positive observations while maintaining a precision of 35.78%. This is a better precision-recall compromise than balanced Logistic Regression.
- HistGradientBoosting also shows relatively low variability across folds, with a ROC-AUC standard deviation of approximately 0.002.
- Based on these cross-validation results, HistGradientBoosting is selected as the candidate model for hyperparameter and decision-threshold optimization.
- The held-out test set has not been used for this selection and remains reserved for final evaluation.